In [0]:
# Load Silver + prepare labels
import mlflow
import mlflow.spark
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.sql import functions as F

skill_gap = spark.table(
    "ecommerce.silver.skill_gap_silver"
).na.fill(0)

skill_gap = skill_gap.withColumn(
    "needs_intervention",
    F.when(
        F.col("intervention_priority")
         .isin(["Critical", "High"]), 1
    ).otherwise(0).cast("int")
)

print("Label distribution:")
skill_gap.groupBy("needs_intervention").count().show()
print(f"Total states: {skill_gap.count()}")

Label distribution:
+------------------+-----+
|needs_intervention|count|
+------------------+-----+
|                 1|    2|
|                 0|   25|
+------------------+-----+

Total states: 27


In [0]:
# Features + split
feature_cols = [
    "avg_unemployment_rate",
    "peak_unemployment_rate",
    "unemployment_volatility",
    "avg_labour_participation",
    "skill_gap_score",
    "months_tracked"
]

skill_gap = skill_gap.fillna(0, subset=feature_cols)

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

train_df, test_df = skill_gap.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_df.count()} | Test: {test_df.count()}")

Train: 20 | Test: 7


In [0]:
# Train + evaluate only 
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline

gbt = GBTClassifier(
    labelCol="needs_intervention",
    featuresCol="features",
    maxIter=50,
    maxDepth=4,
    seed=42
)

pipeline = Pipeline(stages=[assembler, gbt])
model = pipeline.fit(train_df)
preds = model.transform(test_df)

auc = BinaryClassificationEvaluator(
    labelCol="needs_intervention"
).evaluate(preds)

print(f"AUC : {auc:.4f} ✓")
print("Model trained successfully ✓")

AUC : 1.0000 ✓
Model trained successfully ✓


In [0]:
# MLflow metrics only (no model saving)
import mlflow

with mlflow.start_run(run_name="GBT_YouthIntervention_v1"):
    mlflow.log_param("model", "GBTClassifier")
    mlflow.log_param("maxIter", 50)
    mlflow.log_param("maxDepth", 4)
    mlflow.log_metric("AUC", round(auc, 4))
    print("MLflow logged ✓")

MLflow logged ✓


In [0]:
# Score all states → Gold Delta table
all_preds = model.transform(
    skill_gap.fillna(0, subset=feature_cols)
)

all_preds.select(
    "region",
    "avg_unemployment_rate",
    "skill_gap_score",
    "risk_level",
    "intervention_priority",
    "needs_intervention",
    "prediction",
    "probability"
).write.format("delta").mode("overwrite") \
 .saveAsTable("ecommerce.gold.state_predictions_gold")

print("\nTop states needing intervention:")
spark.table("ecommerce.gold.state_predictions_gold") \
    .filter(F.col("prediction") == 1) \
    .orderBy(F.desc("avg_unemployment_rate")) \
    .show(10, truncate=False)

print("\nML + Gold complete ✓")


Top states needing intervention:
+-------+---------------------+---------------+----------+---------------------+------------------+----------+-----------------------------------------+
|region |avg_unemployment_rate|skill_gap_score|risk_level|intervention_priority|needs_intervention|prediction|probability                              |
+-------+---------------------+---------------+----------+---------------------+------------------+----------+-----------------------------------------+
|Haryana|27.48                |15.91          |High      |Critical             |1                 |1.0       |[0.021520885558340797,0.9784791144416592]|
|Tripura|25.06                |10.56          |High      |High                 |1                 |1.0       |[0.021520885558340797,0.9784791144416592]|
+-------+---------------------+---------------+----------+---------------------+------------------+----------+-----------------------------------------+


ML + Gold complete ✓
